# Cài đặt các thư viện cần thiết

In [85]:
# Cài đặt các thư viện cần thiết
%pip install --quiet pulp folium geopy pandas numpy matplotlib openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# Import các thư viện và Cấu hình

In [86]:
import pandas as pd
import numpy as np
from pulp import *
import folium
from geopy.distance import geodesic
import matplotlib.pyplot as plt

# Nhập dữ liệu

Cấu hình dữ liệu để dễ đọc

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

>>> Thư viện đã sẵn sàng!


Thiết lập tham số chung

In [88]:
# --- 1. THAM SỐ TOÀN CỤC (GLOBAL PARAMETERS) ---
SCALE_COST = 1_000_000 # Đơn vị tính: Triệu VND

PARAMS = {
    'Budget_Max': 200_000 * SCALE_COST,   # Ví dụ: 200 Tỷ VND 
    'Charger_Cost': 600 * SCALE_COST,     # 500 Triệu/trụ
    'Charger_Power': 75,                 # 75 kW (DC Fast)
    'Charger_Area': 15,                   # 15 m2/trụ
    'Min_Distance': 0.5,                  # Khoảng cách tối thiểu 0.5 km
    'Max_Charger_Per_Site': 15,           # Tối đa 15 trụ/trạm
    'Interest_Rate': 0.2,                 # Lãi suất 5%
    'Project_Years': 5                    # Vòng đời 5 năm
}

# Tính hệ số Theta (Capital Recovery Factor) theo Ngày
r = PARAMS['Interest_Rate']
n = PARAMS['Project_Years']
theta_year = (r * (1 + r)**n) / ((1 + r)**n - 1)
THETA_DAY = theta_year / 365
print(f"Hệ số Theta (ngày): {THETA_DAY:.6f}")

Hệ số Theta (ngày): 0.000916


In [89]:
# --- 2. ĐỌC FILE DỮ LIỆU ---

# Load file: Districts
df_districts = pd.read_excel('Data/districts.xlsx')
df_districts.columns = df_districts.columns.str.strip()

# Load file: Sites
df_sites = pd.read_excel('Data/sites.xlsx')
df_sites.columns = df_sites.columns.str.strip()

# Load file: Time Profile
df_time = pd.read_excel('Data/time_profile.xlsx')
df_time.columns = df_time.columns.str.strip()

# Data Preprocessing

In [90]:
# --- 3. CHUẨN HÓA TÊN CỘT (QUAN TRỌNG) ---
# Bước này đổi tên cột trong CSV về tên chuẩn tiếng Anh để code bên dưới không bị lỗi.

# Map cho bảng Sites
df_sites = df_sites.rename(columns={
    'ID': 'Site_ID', 'Mã trạm': 'Site_ID',
    'Name': 'Name', 'Tên trạm': 'Name',
    'Lat': 'Lat', 'Vĩ độ': 'Lat',
    'Lon': 'Lon', 'Kinh độ': 'Lon',
    'District_ID': 'District_ID', 'Mã quận': 'District_ID',
    'Rented_land_cost': 'Land_Cost', 'Chi phí đất': 'Land_Cost',
    'Install_Cost': 'Install_Cost', 'Chi phí thi công': 'Install_Cost',
    'Area': 'Area', 'Diện tích': 'Area',
    'Grid_Cap': 'Grid_Cap', 'Công suất lưới': 'Grid_Cap',
    'Base_Load_Peak': 'Base_Load_Peak', 'Tải nền đỉnh': 'Base_Load_Peak'
})

# Map cho bảng Districts
df_districts = df_districts.rename(columns={
    'District_ID': 'District_ID', 'Mã quận': 'District_ID',
    'District_Name': 'District_Name', 'Tên quận': 'District_Name',
    'Total_EV_Demand': 'Total_EV_Demand', 'Nhu cầu sạc': 'Total_EV_Demand'
})

# Map cho bảng Time
df_time = df_time.rename(columns={
    'Hour': 'Hour', 'Giờ': 'Hour',
    'Elec_Price': 'Elec_Price', 'Giá điện': 'Elec_Price',
    'Load_Factor_Res': 'Load_Factor_Res', 'Hệ số tải': 'Load_Factor_Res'
})

def clean_currency(x):
    if isinstance(x, str):
        return float(x.replace(',', '').replace('.', '')) 
    return x


print(">>> Xem trước dữ liệu sau khi chuẩn hóa:")
print("--- SITES ---")
display(df_sites.head(3))
print("--- DISTRICTS ---")
display(df_districts.head(3))
print("--- TIME ---")
display(df_time.head(3))

>>> Xem trước dữ liệu sau khi chuẩn hóa:
--- SITES ---


,Ten_Quan,District_ID,Site_ID,Name,Lat,Lon,District,Land-cost,Land_Cost,Install_Cost,Area,Grid_Cap,Base_Load_Peak
0,Ba Đình,D_02,Site_01,Quảng trường Ba Đình,21.04,105.84,Ba Đình,150000000,1200000000,250000000,80,500,900
1,Ba Đình,D_02,Site_02,Bãi đỗ xe công viên Thủ Lệ,21.03,105.81,Ba Đình,50000000,450000000,200000000,90,400,700
2,Ba Đình,D_02,Site_03,Bãi đỗ xe 81 Láng Hạ,21.02,105.82,Ba Đình,95000000,665000000,220000000,70,400,650


--- DISTRICTS ---


,District_ID,District_Name,Total_EV_Demand
0,D_01,Hoàn Kiếm,"5,460.70"
1,D_02,Ba Đình,"8,934.70"
2,D_03,Tây Hồ,"6,959.50"


--- TIME ---


,Hour,Elec_Price,Load_Factor_Res,EV_Arrival_Factor
0,0,1870,0.20,0.10
1,1,1940,0.30,0.10
2,2,1900,0.20,0.10


In [91]:
# 1. Kiểm tra dữ liệu bị thiếu
if df_sites.isnull().values.any():
    print("⚠️ CẢNH BÁO: Dữ liệu Sites có ô trống (NaN). Hãy kiểm tra file input.")
    df_sites = df_sites.fillna(0) # Điền tạm số 0

# 2. Tạo Dictionary Map: District -> List Sites
district_to_sites = df_sites.groupby('District_ID')['Site_ID'].apply(list).to_dict()

# 3. Tính Ma trận khoảng cách & Xung đột
conflict_pairs = []
site_ids = df_sites['Site_ID'].tolist()
coords = df_sites.set_index('Site_ID')[['Lat', 'Lon']].to_dict('index')

# print("Đang tính toán khoảng cách...")
count_conflict = 0
for i in range(len(site_ids)):
    for j in range(i + 1, len(site_ids)):
        s1, s2 = site_ids[i], site_ids[j]
        
        # Geopy tính khoảng cách theo mặt cầu Trái Đất
        dist = geodesic((coords[s1]['Lat'], coords[s1]['Lon']),
                        (coords[s2]['Lat'], coords[s2]['Lon'])).km
        
        if dist < PARAMS['Min_Distance']:
            conflict_pairs.append((s1, s2))
            count_conflict += 1
            # Uncomment dòng dưới nếu muốn xem chi tiết
            # print(f"  Conflict: {s1} - {s2} ({dist:.2f} km)")

# print(f">>> Hoàn tất. Tìm thấy {count_conflict} cặp trạm vi phạm khoảng cách tối thiểu {PARAMS['Min_Distance']}km.")

# Thiết lập mô hình PuLP

In [92]:
# Tạo mô hình PuLP
mdl = LpProblem("EV_Charging_Hanoi_RealData", LpMinimize)

# --- 1. BIẾN SỐ (VARIABLES) ---
# x[i]: Biến nhị phân (1: Xây, 0: Không)
x = {sid: LpVariable(f"x_{sid}", cat='Binary') for sid in site_ids}

# n[i]: Biến nguyên (Số lượng trụ)
n = {sid: LpVariable(f"n_{sid}", lowBound=0, upBound=PARAMS['Max_Charger_Per_Site'], cat='Integer') 
     for sid in site_ids}

# p[i, t]: Biến thực (Công suất sạc tại giờ t)
p_keys = [(s, t) for s in site_ids for t in df_time['Hour']]
p = {(sid, t): LpVariable(f"p_{sid}_{t}", lowBound=0, cat='Continuous') 
     for sid in site_ids for t in df_time['Hour']}

# --- 2. HÀM MỤC TIÊU (OBJECTIVE) ---
# Z = Chi phí đầu tư (ngày) + Chi phí điện (ngày)
obj = 0

# Chi phí đầu tư cố định
for idx, row in df_sites.iterrows():
    sid = row['Site_ID']
    # Đầu tư hạ tầng & Đất (Gắn với x)
    inv_fixed_daily = (row['Land_Cost'] / 365) + (row['Install_Cost'] * THETA_DAY)
    obj += inv_fixed_daily * x[sid]
    
    # Đầu tư thiết bị (Gắn với n)
    inv_dev_daily = PARAMS['Charger_Cost'] * THETA_DAY
    obj += inv_dev_daily * n[sid]

# Chi phí điện
for sid in site_ids:
    for t in df_time['Hour']:
        price = df_time.loc[t, 'Elec_Price']
        # Chia SCALE_COST để đồng bộ đơn vị tiền tệ (Triệu VND)
        obj += p[(sid, t)] * (price / SCALE_COST)

mdl += obj
print(">>> Hàm mục tiêu đã được thiết lập!")

>>> Hàm mục tiêu đã được thiết lập!


# Thiết lập các constraints

In [93]:
# --- 3. RÀNG BUỘC (CONSTRAINTS) ---

# C1. Ngân sách (Budget)
budget_constraint = 0
for idx, row in df_sites.iterrows():
    sid = row['Site_ID']
    budget_constraint += row['Install_Cost'] * x[sid] + PARAMS['Charger_Cost'] * n[sid]

mdl += budget_constraint <= PARAMS['Budget_Max'], "C1_Budget"

# C2. Lưới điện (Grid Safety)
for idx, row in df_sites.iterrows():
    sid = row['Site_ID']
    grid_cap = row['Grid_Cap'] * 0.95 # An toàn 95%
    base_peak = row['Base_Load_Peak']
    
    for t in df_time['Hour']:
        load_res = base_peak * df_time.loc[t, 'Load_Factor_Res']
        mdl += p[(sid, t)] + load_res <= grid_cap, f"C2_Grid_{sid}_{t}"

# C3. Năng lực trụ sạc (Physical Cap)
for sid in site_ids:
    for t in df_time['Hour']:
        mdl += p[(sid, t)] <= n[sid] * PARAMS['Charger_Power'], f"C3_Phys_{sid}_{t}"

# C4. Logic đầu tư (n <= M*x)
for sid in site_ids:
    mdl += n[sid] <= PARAMS['Max_Charger_Per_Site'] * x[sid], f"C4_Logic_{sid}"

# C5. Diện tích (Area)
for idx, row in df_sites.iterrows():
    sid = row['Site_ID']
    mdl += n[sid] * PARAMS['Charger_Area'] <= row['Area'] * x[sid], f"C5_Area_{sid}"

# C6. Đáp ứng nhu cầu Quận (Demand Satisfaction)
for idx, row in df_districts.iterrows():
    did = row['District_ID']
    demand = row['Total_EV_Demand']
    
    sites = district_to_sites.get(did, [])
    if sites:
        total_p_district = lpSum([p[(s, t)] for s in sites for t in df_time['Hour']])
        mdl += total_p_district >= demand, f"C6_Demand_{did}"
    else:
        print(f"⚠️ Quận {did} không có địa điểm ứng viên nào trong file sites!")

# C7. Khoảng cách (Min Distance)
for (s1, s2) in conflict_pairs:
    mdl += x[s1] + x[s2] <= 1, f"C7_Conflict_{s1}_{s2}"

print(">>> Đã thiết lập xong tất cả ràng buộc!")
# print(f"Số biến: {len(mdl.variables())}")
# print(f"Số ràng buộc: {len(mdl.constraints)}")

>>> Đã thiết lập xong tất cả ràng buộc!


# Giải và Xuất kết quả

In [94]:
# Giải bài toán
print("\nĐang giải bài toán (90 sites × 24 hours)...")
print(f"Kích thước bài toán: {len(site_ids)} sites × {len(df_time)} hours")
print(f"Biến: {len(mdl.variables())} | Ràng buộc: {len(mdl.constraints)}")

# Solver với tối ưu hóa
# - timeLimit=600: 10 phút timeout
# - threads=0: Dùng tất cả threads CPU
# - msg=1: Hiển thị log
# - gapRel=0.05: Chấp nhận nghiệm chỉ cách optimal 5% (tăng tốc độ)
solver = PULP_CBC_CMD(msg=1, timeLimit=600, threads=0, gapRel=0.05)
status = mdl.solve(solver)

def print_resource_stats(df_res, df_sites, PARAMS):
    """Hàm tính và in thống kê tài nguyên"""
    print("\n" + "="*70)
    print("📊 THỐNG KÊ TÀI NGUYÊN SỬ DỤNG")
    print("="*70)
    
    # 1. Số trạm
    num_sites_used = len(df_res)
    num_sites_total = len(df_sites)
    utilization_sites = (num_sites_used / num_sites_total) * 100
    print(f"\n🏢 Trạm sạc:")
    print(f"   • Sử dụng: {num_sites_used} trạm")
    print(f"   • Khả thi: {num_sites_total} trạm")
    print(f"   • Tỷ lệ: {utilization_sites:.1f}%")
    
    # 2. Số trụ sạc
    num_chargers_used = df_res['Chargers'].sum()
    num_chargers_max = num_sites_total * PARAMS['Max_Charger_Per_Site']
    utilization_chargers = (num_chargers_used / num_chargers_max) * 100
    print(f"\n⚡ Trụ sạc:")
    print(f"   • Sử dụng: {num_chargers_used} trụ")
    print(f"   • Khả thi: {num_chargers_max} trụ")
    print(f"   • Tỷ lệ: {utilization_chargers:.1f}%")
    
    # 3. Công suất
    power_used = (df_res['Chargers'] * PARAMS['Charger_Power']).sum()
    power_max = num_chargers_max * PARAMS['Charger_Power']
    utilization_power = (power_used / power_max) * 100
    print(f"\n⚙️  Công suất:")
    print(f"   • Sử dụng: {power_used:,} kW")
    print(f"   • Khả thi: {power_max:,} kW")
    print(f"   • Tỷ lệ: {utilization_power:.1f}%")
    
    # 4. Diện tích
    sites_used = df_res['Site_ID'].tolist()
    area_used = 0
    for s in sites_used:
        area_used += (df_res[df_res['Site_ID']==s]['Chargers'].values[0] * PARAMS['Charger_Area'])
    
    area_total = (df_sites['Area'].sum())
    utilization_area = (area_used / area_total) * 100
    print(f"\n📏 Diện tích:")
    print(f"   • Sử dụng: {area_used:,.0f} m²")
    print(f"   • Khả thi: {area_total:,.0f} m²")
    print(f"   • Tỷ lệ: {utilization_area:.1f}%")
    
    # 5. Chi phí vốn
    total_invest_used = df_res['Invest_Mil_VND'].sum()
    total_invest_max = PARAMS['Budget_Max']
    utilization_cost = (total_invest_used / total_invest_max) * 100
    print(f"\n💰 Vốn đầu tư:")
    print(f"   • Sử dụng: {total_invest_used:,.0f} VND")
    print(f"   • Ngân sách: {total_invest_max:,.0f} VND")
    print(f"   • Tỷ lệ: {utilization_cost:.1f}%")
    
    # 6. Nhu cầu điện
    daily_output = df_res['Daily_Output_kWh'].sum()
    total_demand = df_districts['Total_EV_Demand'].sum()
    demand_satisfaction = (daily_output / total_demand) * 100
    print(f"\n📋 Nhu cầu điện:")
    print(f"   • Cung cấp: {daily_output:,.0f} kWh/ngày")
    print(f"   • Nhu cầu: {total_demand:,.0f} kWh/ngày")
    print(f"   • Thỏa mãn: {demand_satisfaction:.1f}%")
    print("="*70)

if mdl.status == 1:  # LpStatusOptimal = 1
    print(f"\n✅ TÌM THẤY LỜI GIẢI TỐI ƯU!")
    print(f"Objective Value: {value(mdl.objective):,.2f} (VND/ngày)")
    
    # 1. Trích xuất kết quả chi tiết
    results = []
    total_invest_real = 0
    
    for idx, row in df_sites.iterrows():
        sid = row['Site_ID']
        if x[sid].varValue > 0.5: # Trạm được chọn
            num_chargers = int(n[sid].varValue)
            daily_kwh = sum(p[(sid, t)].varValue for t in df_time['Hour'])
            
            # Tính vốn
            invest = row['Install_Cost'] + (num_chargers * PARAMS['Charger_Cost'])
            total_invest_real += invest
            
            results.append({
                'Site_ID': sid,
                'Name': row['Name'],
                'District': row['District_ID'],
                'Lat': row['Lat'],
                'Lon': row['Lon'],
                'Chargers': num_chargers,
                'Daily_Output_kWh': daily_kwh,
                'Invest_Mil_VND': invest
            })
            
    df_res = pd.DataFrame(results)
    
    print(f"\n📊 KẾT QUẢ:")
    print(f"Số trạm được chọn: {len(df_res)}")
    print(f"Tổng vốn đầu tư: {total_invest_real:,.0f} VND")
    print(f"Ngân sách giới hạn: {PARAMS['Budget_Max']:,.0f} VND")
    display(df_res)
    
    # In thống kê tài nguyên
    print_resource_stats(df_res, df_sites, PARAMS)
    
    # 2. Vẽ bản đồ (Visualization)
    print("\n🗺️  Vẽ bản đồ...")
    center_lat = df_sites['Lat'].mean()
    center_lon = df_sites['Lon'].mean()
    m = folium.Map(location=[center_lat, center_lon], zoom_start=12)

    # Điểm KHÔNG được chọn (Đỏ)
    selected_sids = df_res['Site_ID'].tolist()
    for idx, row in df_sites.iterrows():
        if row['Site_ID'] not in selected_sids:
            folium.CircleMarker(
                location=[row['Lat'], row['Lon']],
                radius=3, color='red', fill=True, popup=row['Name']
            ).add_to(m)
            
    # Điểm ĐƯỢC CHỌN (Xanh)
    for idx, row in df_res.iterrows():
        info = f"<b>{row['Name']}</b><br>Trụ: {row['Chargers']}<br>Output: {row['Daily_Output_kWh']:.0f} kWh"
        folium.Marker(
            location=[row['Lat'], row['Lon']],
            popup=folium.Popup(info, max_width=300),
            icon=folium.Icon(color='green', icon='bolt', prefix='fa')
        ).add_to(m)
        
        # Bán kính phủ (800m)
        folium.Circle(location=[row['Lat'], row['Lon']], radius=800, color='green', fill=True, fill_opacity=0.1).add_to(m)
        
    display(m)

else:
    print(f"   Giá trị hiện tại: {value(mdl.objective):,.2f} VND/ngày")
    
    # In kết quả hiện tại 
    results = []
    total_invest_real = 0
    for idx, row in df_sites.iterrows():
        sid = row['Site_ID']
        if x[sid].varValue is not None and x[sid].varValue > 0.5:
            num_chargers = int(n[sid].varValue) if n[sid].varValue is not None else 0
            daily_kwh = sum(p[(sid, t)].varValue for t in df_time['Hour'] if p[(sid, t)].varValue is not None)
            invest = row['Install_Cost'] + (num_chargers * PARAMS['Charger_Cost'])
            total_invest_real += invest
            results.append({
                'Site_ID': sid,
                'Name': row['Name'],
                'District': row['District_ID'],
                'Chargers': num_chargers,
                'Daily_kWh': daily_kwh,
                'Invest_Mil_VND': invest
            })
    
    if results:
        print(f"\n   ✅ Tìm thấy {len(results)} trạm khả thi")
        print(f"   Tổng vốn: {total_invest_real:,.0f} VND")
        
        # Đổi tên cột để khớp với hàm
        for r in results:
            r['Daily_Output_kWh'] = r.pop('Daily_kWh')
        
        df_res = pd.DataFrame(results)
        display(df_res)
        
        # In thống kê tài nguyên
        print_resource_stats(df_res, df_sites, PARAMS)
    else:
        print("   ❌ Không tìm thấy giải pháp khả thi")



Đang giải bài toán (90 sites × 24 hours)...
Kích thước bài toán: 90 sites × 24 hours
Biến: 2340 | Ràng buộc: 4534
   Giá trị hiện tại: 81,138,767.09 VND/ngày

   ✅ Tìm thấy 21 trạm khả thi
   Tổng vốn: 41,520,000,000 VND


,Site_ID,Name,District,Chargers,Invest_Mil_VND,Daily_Output_kWh
0,Site_11,DCAR XANH,D_01,1,780000000,"1,390.00"
1,Site_12,Nhà xe Vấn Quyên,D_01,2,1380000000,"1,680.00"
2,Site_18,Bãi gửi xe ở phố Hoàng Cầu,D_06,3,2000000000,"1,853.34"
3,Site_19,Bãi đỗ xe Trường Chinh,D_06,3,1980000000,"1,775.00"
4,Site_20,Bãi giữ xe đình Kim Liên,D_06,3,1980000000,"1,590.00"
5,Site_21,Công viên Đống Đa,D_06,3,2020000000,"1,471.39"
6,Site_22,P. Chùa Bộc,D_06,2,1440000000,"1,231.67"
7,Site_23,Đại học Y Hà Nội,D_06,5,3220000000,"1,925.00"
8,Site_26,Times City T11,D_07,4,2750000000,"5,438.60"
9,Site_29,Hòa Bình Green City,D_07,3,2100000000,"4,080.00"



📊 THỐNG KÊ TÀI NGUYÊN SỬ DỤNG

🏢 Trạm sạc:
   • Sử dụng: 21 trạm
   • Khả thi: 90 trạm
   • Tỷ lệ: 23.3%

⚡ Trụ sạc:
   • Sử dụng: 62 trụ
   • Khả thi: 1350 trụ
   • Tỷ lệ: 4.6%

⚙️  Công suất:
   • Sử dụng: 4,650 kW
   • Khả thi: 101,250 kW
   • Tỷ lệ: 4.6%

📏 Diện tích:
   • Sử dụng: 930 m²
   • Khả thi: 7,310 m²
   • Tỷ lệ: 12.7%

💰 Vốn đầu tư:
   • Sử dụng: 41,520,000,000 VND
   • Ngân sách: 200,000,000,000 VND
   • Tỷ lệ: 20.8%

📋 Nhu cầu điện:
   • Cung cấp: 89,844 kWh/ngày
   • Nhu cầu: 133,959 kWh/ngày
   • Thỏa mãn: 67.1%


In [95]:
# =============================================================================
# 🗺️ ADVANCED MAP VISUALIZATION - Tất cả các trạm với chi tiết
# =============================================================================

print("\n" + "="*70)
print("🗺️  VISUALIZATION: Bản đồ chi tiết tất cả trạm")
print("="*70)

# Tạo bản đồ căn giữa ở Hà Nội
center_lat = df_sites['Lat'].mean()
center_lon = df_sites['Lon'].mean()

map_all = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=11,
    tiles='OpenStreetMap'
)

# Xác định trạm được chọn vs không được chọn
if 'df_res' in locals() and len(df_res) > 0:
    selected_sids = df_res['Site_ID'].tolist()
else:
    selected_sids = []

# Dữ liệu để tạo heatmap
all_coords = []

# 1. Hiển thị TẤT CẢ các trạm (được chọn màu xanh, không được chọn màu đỏ)
print("\n📍 Thêm các điểm trạm vào bản đồ...")
for idx, row in df_sites.iterrows():
    sid = row['Site_ID']
    lat = row['Lat']
    lon = row['Lon']
    
    # Lưu tọa độ cho heatmap
    all_coords.append([lat, lon])
    
    # Xác định màu sắc và biểu tượng
    if sid in selected_sids:
        # Trạm ĐƯỢC CHỌN - Xanh
        color = 'green'
        icon_color = 'green'
        icon_name = 'bolt'
        status = "✅ ĐƯỢC CHỌN"
        
        # Lấy thông tin chi tiết từ df_res
        res_info = df_res[df_res['Site_ID'] == sid].iloc[0]
        num_chargers = int(res_info['Chargers'])
        daily_output = res_info['Daily_Output_kWh']
        invest = res_info['Invest_Mil_VND']
        
        popup_text = f"""
        <b style='font-size:12px'>{row['Name']}</b><br>
        <hr style='margin:3px 0'>
        <b>{status}</b><br>
        ID: {sid}<br>
        Quận: {row['District_ID']}<br>
        <br>
        <b>📊 Thông số:</b><br>
        • Trụ sạc: <b>{num_chargers}</b> trụ<br>
        • Công suất: <b>{num_chargers * 150}</b> kW<br>
        • Output: <b>{daily_output:,.0f}</b> kWh/ngày<br>
        • Vốn: <b>{invest:,.0f}</b> Triệu VND<br>
        <br>
        <b>🏗️ Cơ sở hạ tầng:</b><br>
        • Diện tích: {row['Area']:,.0f} m²<br>
        • Lưới điện: {row['Grid_Cap']} kW<br>
        • Tải nền: {row['Base_Load_Peak']} kW
        """
        
    else:
        # Trạm KHÔNG ĐƯỢC CHỌN - Đỏ
        color = 'red'
        icon_color = 'red'
        icon_name = 'times'
        status = "❌ KHÔNG ĐƯỢC CHỌN"
        
        popup_text = f"""
        <b style='font-size:12px'>{row['Name']}</b><br>
        <hr style='margin:3px 0'>
        <b>{status}</b><br>
        ID: {sid}<br>
        Quận: {row['District_ID']}<br>
        <br>
        <b>🏗️ Cơ sở hạ tầng:</b><br>
        • Diện tích: {row['Area']:,.0f} m²<br>
        • Lưới điện: {row['Grid_Cap']} kW<br>
        • Tải nền: {row['Base_Load_Peak']} kW<br>
        • Chi phí cơ sở: {row['Install_Cost']:,.0f} VND
        """
    
    # Tạo marker cho trạm
    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(popup_text, max_width=350),
        tooltip=f"{row['Name']} ({status})",
        icon=folium.Icon(
            color=icon_color,
            icon=icon_name,
            prefix='fa'
        )
    ).add_to(map_all)

print(f"   ✓ Đã thêm {len(df_sites)} trạm")

# 2. Thêm heatmap để hiện thị mật độ trạm
print("📈 Thêm Heatmap - Mật độ trạm...")
from folium.plugins import HeatMap
HeatMap(all_coords, radius=15, blur=25, max_zoom=1, min_opacity=0.2).add_to(map_all)

# 3. Thêm vòng phủ sóng cho các trạm được chọn
print("🔆 Thêm vòng phủ sóng...")
if 'df_res' in locals() and len(df_res) > 0:
    for idx, row in df_res.iterrows():
        site_id = row['Site_ID']
        # Truy vấn lại từ df_sites để lấy lat, lon chính xác
        site_info = df_sites[df_sites['Site_ID'] == site_id].iloc[0]
        lat = site_info['Lat']
        lon = site_info['Lon']
        
        # Vòng phủ chính (1000m)
        folium.Circle(
            location=[lat, lon],
            radius=10,
            color='green',
            fill=True,
            fillColor='green',
            fillOpacity=0.15,
            weight=2,
            popup=f"Phủ sóng: {site_info['Name']}"
        ).add_to(map_all)
        
        # Vòng phủ phụ (600m)
        folium.Circle(
            location=[lat, lon],
            radius=600,
            color='darkgreen',
            fill=False,
            weight=1,
            opacity=0.5
        ).add_to(map_all)

# 4. Thêm layer control
from folium import LayerControl
map_all.add_child(LayerControl())

# 5. Thêm title
title_html = '''
             <div style="position: fixed; 
                     top: 10px; left: 50px; width: 350px; height: auto; 
                     background-color: white; border:2px solid grey; z-index:9999; 
                     font-size:13px; font-weight: bold; border-radius: 5px; padding: 10px">
             🗺️ BẢN ĐỒ TRẠM SẠC EV HÀNỘI
             <hr style="margin: 5px 0">
             <span style="color: green">● Xanh: Được chọn ✅</span><br>
             <span style="color: red">● Đỏ: Không được chọn ❌</span><br>
             <span style="color: darkgreen">━ Vòng phủ sóng</span><br>
             <span style="color: orange">▲ Heatmap: Mật độ trạm</span>
             </div>
             '''
map_all.get_root().html.add_child(folium.Element(title_html))

print("✓ Bản đồ hoàn tất!")
display(map_all)

# 6. Thống kê tóm tắt
print("\n" + "="*70)
print("📋 THỐNG KÊ TRẠM")
print("="*70)
print(f"✅ Trạm được chọn: {len(selected_sids)} / {len(df_sites)} ({len(selected_sids)/len(df_sites)*100:.1f}%)")
print(f"❌ Trạm không được chọn: {len(df_sites) - len(selected_sids)} / {len(df_sites)}")

if len(selected_sids) > 0:
    print(f"\n📍 Chi tiết trạm được chọn:")
    for i, sid in enumerate(selected_sids, 1):
        site_info = df_sites[df_sites['Site_ID'] == sid].iloc[0]
        res_info = df_res[df_res['Site_ID'] == sid].iloc[0]
        print(f"\n   {i}. {site_info['Name']} (ID: {sid})")
        print(f"      • Vị trí: {site_info['Lat']:.4f}, {site_info['Lon']:.4f}")
        print(f"      • Trụ sạc: {int(res_info['Chargers'])} trụ")
        print(f"      • Output: {res_info['Daily_Output_kWh']:,.0f} kWh/ngày")
        print(f"      • Vốn: {res_info['Invest_Mil_VND']:,.0f} Triệu VND")



🗺️  VISUALIZATION: Bản đồ chi tiết tất cả trạm

📍 Thêm các điểm trạm vào bản đồ...
   ✓ Đã thêm 90 trạm
📈 Thêm Heatmap - Mật độ trạm...
🔆 Thêm vòng phủ sóng...
✓ Bản đồ hoàn tất!



📋 THỐNG KÊ TRẠM
✅ Trạm được chọn: 21 / 90 (23.3%)
❌ Trạm không được chọn: 69 / 90

📍 Chi tiết trạm được chọn:

   1. DCAR XANH (ID: Site_11)
      • Vị trí: 21.0273, 105.8596
      • Trụ sạc: 1 trụ
      • Output: 1,390 kWh/ngày
      • Vốn: 780,000,000 Triệu VND

   2. Nhà xe Vấn Quyên (ID: Site_12)
      • Vị trí: 21.0214, 105.8616
      • Trụ sạc: 2 trụ
      • Output: 1,680 kWh/ngày
      • Vốn: 1,380,000,000 Triệu VND

   3. Bãi gửi xe ở phố Hoàng Cầu (ID: Site_18)
      • Vị trí: 21.0191, 105.8243
      • Trụ sạc: 3 trụ
      • Output: 1,853 kWh/ngày
      • Vốn: 2,000,000,000 Triệu VND

   4. Bãi đỗ xe Trường Chinh (ID: Site_19)
      • Vị trí: 21.0020, 105.8230
      • Trụ sạc: 3 trụ
      • Output: 1,775 kWh/ngày
      • Vốn: 1,980,000,000 Triệu VND

   5. Bãi giữ xe đình Kim Liên (ID: Site_20)
      • Vị trí: 21.0102, 105.8382
      • Trụ sạc: 3 trụ
      • Output: 1,590 kWh/ngày
      • Vốn: 1,980,000,000 Triệu VND

   6. Công viên Đống Đa (ID: Site_21)
      • Vị trí: 21.0

In [97]:
# Lưu bản đồ ra file HTML
output_file = 'ev_charging_stations_map.html'
map_all.save(output_file)